# 🎙️ AI Video Assistant - Cloud GPU Deployment

This notebook provides a 1-click deployment to run the **AI Video & Meeting Assistant** on Google Colab using NVIDIA T4 GPUs. 

It handles:
1. Cloning the repository
2. Installing Linux audio drivers (`ffmpeg`) and Python dependencies (PyTorch/Whisper)
3. Securely ingesting required API keys
4. Exposing the Streamlit UI via a Cloudflare secure tunnel

In [ ]:
# 1. System Dependencies & Repo Setup
!apt-get update && apt-get install -y ffmpeg -q
!git clone https://github.com/Harsh-00-007/AI-video-assitant.git
%cd AI-video-assitant
!pip install -r requirements.txt -q

# 2. Secure API Key Ingestion
import os
from getpass import getpass
print("\n--- Setup Environment Secrets ---")
os.environ["SARVAM_API_KEY"] = getpass("Enter Sarvam API Key (Press Enter to skip if using local Whisper): ")
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key (For Embeddings/LLM): ")

# 3. Setup Cloudflare Daemon
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 4. Boot Streamlit (Headless)
print("\nBooting Streamlit Application on Port 8501...")
!streamlit run app.py --server.port 8501 &>/content/streamlit_logs.txt &

# 5. Launch Cloudflare Tunnel
import time
time.sleep(5) # Allow Streamlit to spin up before tunneling
print("Establishing Secure Cloudflare Ingress...")
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 &> tunnel.log &
time.sleep(5)

# 6. Extract Public URL
print("\n" + "="*50)
print("✅ YOUR AI ASSISTANT IS LIVE!")
print("🔗 Access your app here: ")
!grep -o 'https://.*\.trycloudflare.com' tunnel.log
print("="*50 + "\n")